In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_receiving_year(year):
    url = f"https://www.sports-reference.com/cfb/years/{year}-receiving.html"
    print(f"Scraping {year}...")
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        table = soup.find("table", id="receiving_standard")
        if not table:
            print(f"[!] Table not found for {year}. Skipping.")
            return

        tbody = table.find("tbody")
        first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
        columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

        data = []
        for row in tbody.find_all("tr"):
            if row.get("class") and "thead" in row.get("class"):
                continue
            tds = row.find_all("td")
            if not tds:
                continue
            values = [td.get_text(strip=True) for td in tds]
            data.append(values)

        df = pd.DataFrame(data, columns=columns)

        for col in df.columns:
            if df[col].dtype == object:
                df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

        df.to_csv(f"cfb_{year}_receiving.csv", index=False)
        print(f"✅ Saved: cfb_{year}_receiving.csv")

        time.sleep(1.5)  # be polite to the server
    except Exception as e:
        print(f"[!] Error for {year}: {e}")

if __name__ == "__main__":
    for year in range(2024, 2011, -1):  # From 2024 to 2012 inclusive
        scrape_receiving_year(year)
